# HTML Scraping

**MGS 4701 W01** | Week 3-1 | Tuesday, 15 September 2026

|   |Topics | Min |
| --- | --- | --- |
| 0 | Review: what an API gave us | 8 |
| 1 | Two URLs, two kinds of answer | 5 |
| 2 | Parsing HTML — a BA master's programme page | 12 |
| 3 | Following links inside a page | 12 |
| 4 | When the page is dynamic | 8 |
| 5 | Selenium | 12 |
| 6 | What we are not covering | 6 |
|   | **Exercise — scrape a dynamic site** | 12 |

---
## 0. Review — last session

### An API gives you two things: **data** and **services**

Last Thursday we used the **GitHub REST API** to get *data* — repositories matching our keywords, returned as structured records.

**Has anyone used an LLM API?** OpenAI, Claude, DeepSeek, Qwen?

That is the other kind. When you call OpenAI's API you are not downloading a dataset — there is no file of answers sitting on their server waiting for you.
You send text, a model runs on their hardware, and generated text comes back. You are buying **computation**, not data. Same mechanism, different product:

| | You send | You get back | You are paying for |
| --- | --- | --- | --- |
| GitHub API | a search query | records that already existed | their database |
| OpenAI / Claude API | a prompt | text generated on demand | the model weights |

Other service APIs: payment (Stripe, Alipay), maps and routing, translation, speech-to-text, sending email.

### How you call one

```python
response = requests.get(url, headers=headers, params=params, timeout=20)
```

```python
API_URL   = "https://api.github.com/search/repositories"
HEADERS = {"Authorization": f"Bearer {GITHUB_TOKEN}", "Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28", "User-Agent": "MGS4701-teaching-notebook",}
params = {"q": "data analyst interview", "sort": "updated", "order": "desc", "per_page": 5, "page": 1,}
response = requests.get(API_URL, headers=HEADERS, params=params, timeout=20)
```

**In:** the URL (which resource), headers (who you are — your token), [params](https://requests.readthedocs.io/en/latest/user/quickstart/#passing-parameters-in-urls)(the filters), timeout (how long `requests` waits before giving up).

>https://api.github.com/search/repositories?q=data+analyst+interview&sort=updated&order=desc&per_page=5&page=1

**Out:** A single `Response` object
- Attributes of an `Response` object: 
  - `.status_code`: 200-OK, 401-Unauthorized, 403-Forbidden, 404-Not Found
  - `.headers`: what is left of your rate limit
  - `.text`: the body
  - `.url`: the final URL, with params appended
- Methods of an `Response` object:
  - `.json()`: the body parsed, **only if it is a valid JSON**

### So: where should you get data from?

1. **Is there an official API?** Search *"[platform] API documentation"*, or look for "Developers" in the site footer. If yes, use it — that is last Thursday.
2. **Has someone already published the dataset?** Kaggle, GitHub, government open-data portals. Cheapest of all, *but* you inherit someone else's decisions. Before you use it, ask: who collected it, when, from where, and
   how? A dataset with no documented provenance cannot support a claim in your report, however convenient it is.
3. **No API, no dataset?** Read `robots.txt`, then scrape the HTML. **That is today.**

### The one difference that matters today

| | Returns | You get |
| --- | --- | --- |
| An **API** endpoint | JSON | structured records, typed fields, ready to use |
| An **ordinary** page URL | HTML | a document written to be *displayed*, not read by a program |

Nobody designed that HTML for you. **First challenge: parsing it.**

In [1]:
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "MGS4701 teaching exercise (dchen@kean.edu)"}
print("ready")

ready


---
## 1. Two URLs, two kinds of answer

Same function, same arguments. Look at what comes back.

In [2]:
api  = requests.get("https://api.github.com/repos/pandas-dev/pandas", headers=HEADERS, timeout=20)
page = requests.get("https://www.ntu.edu.sg/education/graduate-programme/master-of-science-in-business-analytics", headers=HEADERS, timeout=20)

print(f"API  : {api.headers['content-type'][:30]:32} {len(api.text):>8,} chars")
print(f"PAGE : {page.headers['content-type'][:30]:32} {len(page.text):>8,} chars")

print("\nAPI, first 120 chars:\n ", api.text[:120])
print("\nPAGE, first 120 chars:\n ", page.text[:120])

API  : application/json; charset=utf-      6,523 chars
PAGE : text/html; charset=utf-8          157,892 chars

API, first 120 chars:
  {"id":858127,"node_id":"MDEwOlJlcG9zaXRvcnk4NTgxMjc=","name":"pandas","full_name":"pandas-dev/pandas","private":false,"o

PAGE, first 120 chars:
   <!DOCTYPE html> <html class='no-js' lang='en'> <head> <meta charset="utf-8"><script type="text/html" id="sf-tracking-co


- The API answer is small and already organised — `api.json()["stargazers_count"]` and you are done. 
- The page is hundreds of thousands of characters of layout, navigation and scripts, with the four facts you want buried somewhere inside.

**That is the job: find them.**

---
## 2. Parsing HTML — a BA master's programme page

Our target: **[NTU MSc Business Analytics](https://www.ntu.edu.sg/education/graduate-programme/master-of-science-in-business-analytics)**. Relevant to every one of you this semester, and a page with no API behind it.

Permission first (`host/robots.txt`), then the static test.

In [3]:
URL = ("https://www.ntu.edu.sg/education/graduate-programme/master-of-science-in-business-analytics")


def check_robots(url, agent=HEADERS["User-Agent"], show=10):
    """Fetch robots.txt, show the rules, return True / False / None."""
    robots_url = urljoin(url, "/robots.txt")
    try:
        r = requests.get(robots_url, headers=HEADERS, timeout=10)
    except requests.RequestException as e:
        print(f"could not reach {robots_url}: {type(e).__name__}")
        return None
    if r.status_code == 404:
        print("no robots.txt published — no rules to follow")
        return True
    if r.status_code >= 400:
        print(f"robots.txt returned {r.status_code} — cannot tell")
        return None

    keep = ("user-agent", "disallow", "allow", "crawl-delay")
    rules = [ln.strip() for ln in r.text.splitlines()
             if ln.strip().lower().startswith(keep)]
    print(f"{robots_url}  ({len(rules)} rules)")
    for line in rules[:show]:
        print("   ", line)
    if len(rules) > show:
        print(f"    ... {len(rules) - show} more")

    rp = RobotFileParser()
    rp.parse(r.text.splitlines())
    verdict = rp.can_fetch(agent, url)
    print(f"\nPython's parser says: {verdict}")

    wild = sorted({x for x in rules
                   if x.lower().startswith(("disallow", "allow"))
                   and ("*" in x or "$" in x)})
    if wild:
        print(f"\n!! {len(wild)} rules use * or $, which Python's parser IGNORES.")
        print("   Check your URL against these yourself, e.g.:")
        for x in wild[:6]:
            print("      ", x)
    return verdict


allowed = check_robots(URL)


https://www.ntu.edu.sg/robots.txt  (3 rules)
    User-agent: *
    Disallow: /CustomWebForms
    Disallow: /Sitefinity/*

Python's parser says: True

!! 1 rules use * or $, which Python's parser IGNORES.
   Check your URL against these yourself, e.g.:
       Disallow: /Sitefinity/*


**Try:** `https://www.ntu.edu.sg/CustomWebForms/`, what do you get?

In [4]:
# Now the static test: is the content in the first response?
page.encoding = page.apparent_encoding
print("Is the fee in the HTML?   ", "75,210" in page.text)
print("Is a course name in it?   ", "Analytics Strategy" in page.text)


Is the fee in the HTML?    True
Is a course name in it?    True


Both `True` → the content arrived in the first response. **Static.** `requests` is enough and we never need a browser.

In [5]:
soup = BeautifulSoup(page.text, "html.parser")
text = soup.get_text(" ", strip=True)

h1 = soup.find("h1")
print("Title :", h1.get_text(strip=True) if h1 else "!! no <h1> found")
print("Fees  :", re.findall(r"S\$[\d,]+(?:\.\d{2})?", text)[:3] or "!! none")
print("Dates :", re.findall(r"Round \d: \d{1,2} \w+", text)[:3] or "!! none")
print("Links :", len([a for a in soup.select("a[href]")
                      if "/education/graduate-programme/" in a["href"]]))

Title : Master of Science in Business Analytics
Fees  : ['S$75,210.00', 'S$100']
Dates : ['Round 1: 30 November', 'Round 2: 31 January', 'Round 3: 31 March']
Links : 5


In [6]:
text

"NTU Master of Science in Business Analytics | NTU Singapore Toggle notification Toggle search Toggle menu About Us Back Back About Us The Chancellery Back The Chancellery President Tharman Shanmugaratnam Mr S. Chandra Das Ms Jennie Chua Dr Chua Thian Poh Prof Yaacob bin Ibrahim Board of Trustees University Leadership Provost Office's Appointment Holders Our History Facts & Figures Back Facts & Figures University Rankings (World / Asia) NTU at a Glance Enrolment by College Research Grants / Revenue Faculty and Staff Population Undergraduate Population Graduate Student Population Graduate Output - First Degree & Higher Degree Careers Annual Reports Global Back Global Networks & Alliances Global Partnerships For Students Visit Us Sustainability Back Sustainability Our Commitment Back Our Commitment Governance Social Impact Sustainability Education Back Sustainability Education Undergraduate Programmes Postgraduate Programmes Continuing Education CIFAL Sustainability Research Green Campus

Note what we did **not** do: hunt for a CSS class like `div.programme-fee`. University sites redesign, and class names change with them. A pattern in the *text* — `S$` followed by digits — survives a redesign. Use the most stable handle available, not the first one you find.

In [7]:
def parse_programme(url):
    """One programme page -> one row."""
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.encoding = r.apparent_encoding
    s = BeautifulSoup(r.text, "html.parser")
    t = s.get_text(" ", strip=True)

    fees = re.findall(r"S\$[\d,]+(?:\.\d{2})?", t)
    h1 = s.find("h1")
    return {
        "programme":  h1.get_text(strip=True) if h1 else "",
        "fee":        fees[0] if fees else "",
        "gmat_gre":   "GMAT or GRE score is optional" in t,
        "deadlines":  " | ".join(re.findall(r"Round \d: \d{1,2} \w+", t)),
        "source_url": url,
    }


parse_programme(URL)

{'programme': 'Master of Science in Business Analytics',
 'fee': 'S$75,210.00',
 'gmat_gre': True,
 'deadlines': 'Round 1: 30 November | Round 2: 31 January | Round 3: 31 March',
 'source_url': 'https://www.ntu.edu.sg/education/graduate-programme/master-of-science-in-business-analytics'}

---
## 3. Links inside the page

One programme is one row. A table needs many. The page already tells us where the others are — the **Related Programmes** block at the bottom.

A link is `<a href="...">`. Collect them by **URL pattern**, not by class name.

In [8]:
links = {
    urljoin(URL, a["href"])
    for a in soup.select("a[href]")
    if "/education/graduate-programme/" in a["href"]
}
links.discard(URL)

print(f"{len(links)} other programme pages found\n")
for u in sorted(links)[:8]:
    print(" ", u.rsplit("/", 1)[-1])

5 other programme pages found

  master-of-science-in-accountancy
  master-of-science-in-actuarial-and-risk-analytics
  master-of-science-in-asset-wealth-management
  master-of-science-in-financial-engineering2
  master-of-science-in-marketing-science


### When does it stop?

Each of those pages links onward to its own related programmes. Follow blindly and you will walk the whole university — thousands of pages, hours of requests, and a dataset you cannot describe.

Three ways to stop, in order of preference:

| Rule | Example |
| --- | --- |
| **A fixed list** you chose | these five programmes, named in advance |
| **Depth** | start page + one hop, no further |
| **A pattern** | only URLs containing `/graduate-programme/` |

We use all three: depth 1, this URL pattern, and we keep what we get.
<span style="color:red">**Never write a crawler without a stopping rule.**</span>

In [9]:
rows = [parse_programme(URL)]

for u in sorted(links)[:5]:                 # depth 1, five pages
    rows.append(parse_programme(u))
    time.sleep(1)                           # rule 2 — one request per second

programmes = pd.DataFrame(rows)
programmes

,programme,fee,gmat_gre,deadlines,source_url
0,Master of Science in Business Analytics,"S$75,210.00",True,Round 1: 30 November | Round 2: 31 January | R...,https://www.ntu.edu.sg/education/graduate-prog...
1,Master of Science in Accountancy,"S$70,850.00",False,Round 1: 30 November | Round 2: 31 January | R...,https://www.ntu.edu.sg/education/graduate-prog...
2,Master of Science in Actuarial and Risk Analytics,"S$70,850.00",True,Round 1: 30 November | Round 2: 31 January | R...,https://www.ntu.edu.sg/education/graduate-prog...
3,Master of Science in Asset & Wealth Management,"S$85,020",False,,https://www.ntu.edu.sg/education/graduate-prog...
4,Master of Science in Financial Engineering,"S$8,000",False,Round 1: 30 November | Round 2: 31 January | R...,https://www.ntu.edu.sg/education/graduate-prog...
5,Master of Science in Marketing Science,"S$73,030.00",True,Round 1: 30 November | Round 2: 31 January | R...,https://www.ntu.edu.sg/education/graduate-prog...


Six rows, one source, a few seconds. Note the blanks — not every programme
states a fee on its page. **A blank is a finding, not a bug**, provided you
checked one by eye before deciding that.

---
## 4. When the page is dynamic

Everything above worked because the content was in the first response. Now a page where it is not.

Open <https://quotes.toscrape.com/scroll> in your browser. You see quotes. Scroll, and more appear. Now watch what Python gets.

In [10]:
dyn = requests.get("https://quotes.toscrape.com/scroll", headers=HEADERS, timeout=15)
dsoup = BeautifulSoup(dyn.text, "html.parser")

print(f"status {dyn.status_code}, {len(dyn.text):,} chars")
print(f"quotes found in the HTML: {len(dsoup.select('.quote'))}")
print("\nIs 'Albert Einstein' in what the server sent?", "Albert Einstein" in dyn.text)

status 200, 2,671 chars
quotes found in the HTML: 0

Is 'Albert Einstein' in what the server sent? False


In [18]:
dsoup

<!DOCTYPE html>

<html lang="en">
<head>
<meta charset="utf-8"/>
<title>Quotes to Scrape</title>
<link href="/static/bootstrap.min.css" rel="stylesheet"/>
<link href="/static/main.css" rel="stylesheet"/>
</head>
<body>
<div class="container">
<div class="row header-box">
<div class="col-md-8">
<h1>
<a href="/" style="text-decoration: none">Quotes to Scrape</a>
</h1>
</div>
<div class="col-md-4">
<p>
<a href="/login">Login</a>
</p>
</div>
</div>
<div class="row">
<div class="col-md-8">
<div class="quotes"></div>
</div>
</div>
<div id="loading" style="background-color: #eeeecc"><h5>Loading...</h5></div>
<script src="/static/jquery.js"></script>
<script>
    $(function(){
        var page = 1, tag = null, hasNextPage = true;
        function appendQuotes(quotes) {
            var $quotes = $('.quotes');
            var html = $.map(quotes, function(d){
                var tags = $.map(d['tags'], function(t) {
                    return "<a class='tag'>" + t + "</a>";
                }).jo

**200 OK, and zero quotes.** The request succeeded. The data is missing.

<span style="color:red">That is what *dynamic* means: the server sent a near-empty shell, and JavaScript filled it in **after** the page loaded, inside your browser. Python does not run JavaScript, so Python sees the shell.</span>

### Before reaching for a browser, look for the door

The page fetched those quotes from somewhere. Find out where: **DevTools → Network → XHR →  Reload page.** A request to `quotes?page=1` and `quotes?page=2` appears. Click one, check the Headers, **an API url**!!

The "dynamic page problem" is usually an API problem in disguise.

In [11]:
quotes, page_no = [], 1
while True:
    r = requests.get("https://quotes.toscrape.com/api/quotes", params={"page": page_no}, headers=HEADERS, timeout=15)
    data = r.json()
    quotes += [{"author": q["author"]["name"], "text": q["text"][:60]} for q in data["quotes"]]
    if not data["has_next"]:
        break
    page_no += 1
    time.sleep(0.5)

print(f"{len(quotes)} quotes from {page_no} pages, no browser needed")
pd.DataFrame(quotes).head(3)

100 quotes from 10 pages, no browser needed


,author,text
0,Albert Einstein,“The world as we have created it is a process ...
1,J.K. Rowling,"“It is our choices, Harry, that show what we t..."
2,Albert Einstein,“There are only two ways to live your life. On...


Cleaner than scraping the rendered page would have been, and faster. Note the
stopping rule came from the data itself — `has_next` — not from a number we
guessed.

---
## 5. Selenium — when there is no door

Sometimes there is no convenient endpoint. Then you drive a real browser: Selenium opens Chrome, waits for JavaScript to finish, and hands you the finished HTML.

Selenium 4 downloads the driver itself; you need Chrome installed. This is the **last resort** — slow, heavy, and it breaks when the browser updates.

In [12]:
# pip install selenium

In [13]:
# DEMO — instructor runs this. Watch a browser open, load, and hand back HTML.
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

opts = Options()
opts.add_argument("--headless=new")        # no visible window
driver = webdriver.Chrome(options=opts)

driver.get("https://quotes.toscrape.com/js/")
time.sleep(2)                              # let the JavaScript run

rendered = BeautifulSoup(driver.page_source, "html.parser")
print(f"quotes after rendering: {len(rendered.select('.quote'))}")
for q in rendered.select(".quote")[:2]:
    print(" ", q.select_one(".author").text, "—", q.select_one(".text").text[:50])

driver.quit()

quotes after rendering: 10
  Albert Einstein — “The world as we have created it is a process of o
  J.K. Rowling — “It is our choices, Harry, that show what we truly


`driver.page_source` is the HTML **after** JavaScript ran — then BeautifulSoup as usual. Nothing about parsing changes; only how you obtained the document.

**The costs.** Each page opens a full browser: ~100× slower than `requests`, heavy on memory, and `time.sleep(2)` is a guess. A slow connection returns an empty page and your scraper reports success. Proper code waits for an element to appear (`WebDriverWait`) rather than sleeping.

**Decision order for any dynamic page:** the page's own API → a sitemap or static version → Selenium.

---
## 6. What we are not covering

Real sites defend themselves. Know the names so you recognise the wall:

| | Problem | Usual response |
| --- | --- | --- |
| **Login** | data sits behind an account | `requests.Session()` keeps cookies across requests |
| **Cookies & consent** | a banner blocks the content | the site is asking for agreement; clicking it programmatically is accepting terms on your behalf |
| **CAPTCHA** | "verify you are human" | an explicit refusal of automated traffic. **Stop.** Never solve or outsource one |
| **Rate limits** | `429 Too Many Requests` | slow down, back off exponentially. Usually you caused it |
| **IP blocking** | works, then 403s forever | you were too fast. Rotating proxies is evasion, not a fix |
| **Bot fingerprinting** | headless browsers detected | the site can tell Selenium from Chrome |
| **Infinite scroll / lazy loading** | only 10 of 500 rows appear | find the underlying API, as in section 4 |
| **Silent breakage** | columns quietly go empty | the markup changed. Nobody will tell you — check row counts every run |
| **Terms of service** | `robots.txt` allows it, the ToS does not | two different documents. Read both |

Four of these nine are the site saying no. **Recognising a refusal is a skill, and respecting it is a rule.**

---
## 🖐 Exercise — two loading patterns (12 min)

`quotes.toscrape.com` is a practice site built for exactly this. The same quotes are served four ways, one per path:

| Path | How the page loads | What `requests` alone gets you |
| --- | --- | --- |
| `/` | **Static + pagination.** All ten quotes are in the HTML. A `Next →` link leads to `/page/2/`, and so on to page 10. | Everything — follow the link, repeat |
| `/scroll` | **Infinite scroll.** The shell arrives empty; JavaScript appends more quotes as you scroll. | Nothing (this was §4) |
| `/js` | **JavaScript-rendered.** The shell arrives empty; a script writes the quotes in once, on load. | Nothing |
| `/login` | **Behind a session.** Needs a CSRF token and a logged-in cookie. | A login form — and our rule says stop here |

Two of these are your task.

**Part A — pagination (`/`).** We ran out of time for this last session, so it is new. Collect **author** and **tags** for *every* quote across *all* pages, not just the first. There are ten pages and 100 quotes.

The rule that matters: stop when the site says to stop. Page 10 has no `Next →` link — let its absence end your loop. Do not write `range(1, 11)`; the day the site adds a page, that code silently under-collects.

**Part B — dynamic (`/js`).** Same fields, first page only. `requests` returns nothing here, so either:
- **Selenium** — as in §5, then parse `driver.page_source`
- **No Selenium** — the static twin at `/` holds the same data. Use it, and say in a comment why that is legitimate here and would not be on a site with no static version

Start by proving the problem exists.

In [14]:
# Step 1 — confirm it is dynamic
check = requests.get("https://quotes.toscrape.com/js/", headers=HEADERS, timeout=15)
print("quotes visible to requests:",
      len(BeautifulSoup(check.text, "html.parser").select(".quote")))

quotes visible to requests: 0


In [15]:
# Step 2A — pagination. Fill this in.
paged = []                                  # list of {"author": ..., "tags": ...}
url   = "https://quotes.toscrape.com/"

while url:
    # TODO: fetch url, soup it
    # TODO: for each .quote, append {"author": ..., "tags": ...}
    # TODO: find the next link -> soup.select_one("li.next a")
    # TODO: if it exists, url = urljoin(url, link["href"]); else url = None
    break                                   # remove this once the loop works
    time.sleep(0.5)                          # one request per half-second

pages_df = pd.DataFrame(paged)
print(len(pages_df), "quotes")
pages_df.head()

0 quotes


""


In [16]:
# Step 2B — the dynamic page. Fill this in.
my_quotes = []          # list of {"author": ..., "tags": ...}

# TODO: get the rendered HTML (Selenium) or use the static twin
# TODO: for each .quote, read .author and the .tag elements
# TODO: append one dict per quote

qdf = pd.DataFrame(my_quotes)
qdf.head()

""


In [17]:
# Part A
assert len(pages_df) == 100, f"expected 100 quotes across 10 pages, got {len(pages_df)}"
assert pages_df["author"].nunique() > 10, "too few distinct authors — are you refetching page 1?"

# Part B
assert len(qdf) == 10, f"expected 10 quotes on page 1, got {len(qdf)}"
assert set(qdf.columns) >= {"author", "tags"}, "need author and tags columns"
assert qdf["author"].notna().all() and (qdf["author"] != "").all(), \
    "some authors are empty — check your selector"
assert qdf["tags"].str.len().gt(0).any(), "tags are empty on every row"
print("✓ passed")
qdf.head()

AssertionError: expected 100 quotes across 10 pages, got 0

<details><summary>Hint — parsing one quote</summary>

Each quote is `<div class="quote">` containing `<small class="author">` and
several `<a class="tag">`. Tags are a **list**, so join them into a string before
they reach the DataFrame — `"; ".join(t.text for t in q.select(".tag"))`.
</details>

<details><summary>Hint — the pagination loop</summary>

The link you want is `<li class="next"><a href="/page/2/">`. So:

```python
nxt = soup.select_one("li.next a")
url = urljoin(url, nxt["href"]) if nxt else None
```

`urljoin` turns the relative `/page/2/` into a full URL. When `nxt` is `None`
— which happens on page 10 — `url` becomes `None` and the `while` loop ends on
its own. That is the stopping rule coming from the site rather than from you.
</details>

---
## To do

1. **Thursday is mock interview round 1.** Read the instructions on Canvas before you come.

2. **Start collecting.** Run the static test on your track's main source: fetch it and search the HTML for a phrase you can see on screen. You will know within a minute whether you are in section 2 territory or section 4.

3. **And the part that is not technical.** Scraping gives you rows. Rows are not an answer. Before you write the loop, settle what question the data is meant to answer — then check that the fields you are collecting can actually answer it. A clean dataset aimed at nothing is the most common way this assignment goes wrong.

4. **A1 — project brief and pilot data — is due Monday 28 September.**